# 04 Backtest Results

Run the walk-forward backtest and review out-of-sample performance.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import plotly.express as px
from config import settings
from config.logging_config import configure_logging
from src.data.fetcher import fetch_fx_data
from src.features.builder import build_features
configure_logging(level="WARNING")


In [ ]:
raw = fetch_fx_data(start="2016-01-01")
features = build_features(raw, persist=False)

In [ ]:
from src.backtest.engine import run_backtest
from src.backtest.metrics import compute_metrics
result = run_backtest(features, include_autoencoder=False)
metrics = compute_metrics(result.strategy_returns, result.equity_curve, result.trades)
pd.Series(metrics.as_dict())

## Equity curve

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=result.equity_curve.index, y=result.equity_curve.values, name="Strategy"))
fig.add_trace(go.Scatter(x=result.benchmark_equity.index, y=result.benchmark_equity.values, name="Benchmark"))
fig

## Commentary

The strategy is a defensive overlay: it shorts or reduces exposure on confirmed anomalies. Evaluate Sharpe, max drawdown, and the per-pair breakdown below before drawing conclusions. Past performance does not guarantee future results.

In [ ]:
result.trades.groupby("pair")["trade_return"].agg(["size", "sum", "mean"])